# Clustering experiment: how should HDB blocks be grouped into neighbourhoods?

**What this was for.** The proposal grouped HDB blocks into neighbourhoods with a "blocks within
200–300 m of each other form a cluster" rule (DBSCAN / single linkage). Before building the
production layer, several distance-based methods were run on all residential block centroids,
per subzone, and compared on the error that matters for the study: the **centroid error**, how far
a block sits from its cluster's unit-weighted centroid, i.e. the walking distance we would misstate
if one walk time per cluster stood in for every block.

**What it found** (`docs/clustering_decision.md`). Chaining fails its own purpose: HDB estates are
so dense that DBSCAN at 250 m produces 193 clusters nationwide, one of them 413 blocks spanning
all of Tampines East, with a mean centroid error of about 370 m (~5 min walk). Complete-linkage
agglomerative clustering with a 300 m threshold caps every cluster's diameter: about 1,400 clusters
(median 7 blocks, ~790 flats, which is roughly an HDB precinct), mean centroid error ~80 m, 95th
percentile ~140 m. The 400 m variant (981 clusters) is the fallback.

**Decision it informed.** Complete linkage, 300 m, per subzone (Minh, 2026-10-18), implemented in
[`../1_Data_Preparation/1_03_cluster_hdb_blocks.ipynb`](../1_Data_Preparation/1_03_cluster_hdb_blocks.ipynb).
Since the 2026-10-26 consultation the walk matrix uses individual blocks as origins, so the
clusters serve mapping only and the centroid error no longer enters the accessibility figures.
Original script: `scripts/experiment_clustering.py` (Minh).

**Inputs**
- `output/hdb_blocks.gpkg`, layers `hdb_blocks` (1_02) and `subzones` (1_01).

**Outputs**
- `output/clustering_experiment.csv` — one row per configuration (11), the table behind the
  decision note.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
from scipy.spatial.distance import pdist
from sklearn.cluster import DBSCAN, AgglomerativeClustering

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "bt4015").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from bt4015.paths import DATA, OUT, FIG, LARGE

GPKG = OUT / "hdb_blocks.gpkg"
OUT_CSV = OUT / "clustering_experiment.csv"

## 1. Residential block centroids by subzone

Same block list as the production notebook: residential blocks with dwelling units, centroids in
metres, tagged with their subzone (boundary cases snapped to the nearest subzone).

In [2]:
# 1. Residential blocks with units, as centroids.
blocks = pyogrio.read_dataframe(GPKG, layer="hdb_blocks")
blocks = blocks[blocks["is_residential"] & blocks["total_dwelling_units"].gt(0)].copy()
blocks["geometry"] = blocks.geometry.centroid

# 2. Subzone tag; centroids on a boundary snap to the nearest subzone.
sz = pyogrio.read_dataframe(GPKG, layer="subzones")[["SUBZONE_C", "SUBZONE_N", "PLN_AREA_N", "geometry"]]
pts = gpd.sjoin(blocks, sz, how="left", predicate="within")
missing = pts["SUBZONE_C"].isna().sum()
if missing:
    nn = gpd.sjoin_nearest(pts[pts["SUBZONE_C"].isna()].drop(columns=["index_right", "SUBZONE_C", "SUBZONE_N", "PLN_AREA_N"]), sz, how="left")
    pts.loc[nn.index, ["SUBZONE_C", "SUBZONE_N", "PLN_AREA_N"]] = nn[["SUBZONE_C", "SUBZONE_N", "PLN_AREA_N"]].values
print(f"{len(pts):,} residential blocks with units; {missing} needed nearest-subzone snap; "
      f"{pts['SUBZONE_C'].nunique()} subzones with HDB")

X_all = np.c_[pts.geometry.x, pts.geometry.y]
w_all = pts["total_dwelling_units"].to_numpy(float)

10,769 residential blocks with units; 0 needed nearest-subzone snap; 159 subzones with HDB


## 2. The configurations compared

Three families: DBSCAN with `min_samples=1` (pure single-link chaining, the proposal's rule) at
150–300 m; complete-linkage agglomerative clustering (a hard cap on diameter) at 250–500 m; and
average linkage at 200–400 m as a middle ground. Each is run inside every subzone.

In [3]:
CONFIGS = []
for eps in (150, 200, 250, 300):
    CONFIGS.append((f"dbscan_{eps}m", "DBSCAN (single-link chaining)", eps,
                    lambda X, e=eps: DBSCAN(eps=e, min_samples=1).fit_predict(X)))
for th in (250, 300, 400, 500):
    CONFIGS.append((f"complete_{th}m", "Agglomerative complete (max diameter)", th,
                    lambda X, t=th: AgglomerativeClustering(n_clusters=None, distance_threshold=t, linkage="complete").fit_predict(X)))
for th in (200, 300, 400):
    CONFIGS.append((f"average_{th}m", "Agglomerative average", th,
                    lambda X, t=th: AgglomerativeClustering(n_clusters=None, distance_threshold=t, linkage="average").fit_predict(X)))


def run(label, fn):
    """Cluster ids for every block, fitting fn separately inside each subzone."""
    cid = np.empty(len(pts), dtype=object)
    for code_, idx in pts.groupby("SUBZONE_C").indices.items():
        X = X_all[idx]
        lab = fn(X) if len(idx) > 1 else np.zeros(1, int)
        cid[idx] = [f"{code_}_{l}" for l in lab]
    return cid


print(f"{len(CONFIGS)} configurations")

11 configurations


## 3. Run every configuration and measure it

Per configuration: number of clusters, size distribution (blocks and flats), diameter, and the
centroid error weighted by dwelling units (the share of flats more than 300 m from their cluster
centroid is the "would be misrepresented" share).

In [4]:
rows, labels = [], {}
for label, family, param, fn in CONFIGS:
    # 1. Cluster, then summarise per cluster.
    cid = run(label, fn)
    labels[label] = cid
    df = pd.DataFrame({"cid": cid, "x": X_all[:, 0], "y": X_all[:, 1], "w": w_all})
    g = df.groupby("cid")
    sizes = g.size()
    units = g["w"].sum()
    # 2. Unit-weighted cluster centroid and each block's distance to it (the centroid error).
    cx = g.apply(lambda d: np.average(d.x, weights=d.w), include_groups=False)
    cy = g.apply(lambda d: np.average(d.y, weights=d.w), include_groups=False)
    err = np.hypot(df.x - df.cid.map(cx), df.y - df.cid.map(cy))
    diam = g.apply(lambda d: pdist(d[["x", "y"]].to_numpy()).max() if len(d) > 1 else 0.0, include_groups=False)
    rows.append({
        "config": label, "family": family, "param_m": param,
        "n_clusters": len(sizes),
        "singletons": int((sizes == 1).sum()),
        "median_blocks": float(sizes.median()), "max_blocks": int(sizes.max()),
        "median_units": float(units.median()), "max_units": int(units.max()),
        "median_diam_m": float(diam.median()), "p95_diam_m": float(diam.quantile(.95)), "max_diam_m": float(diam.max()),
        "mean_err_m": float(np.average(err, weights=w_all)),
        "p95_err_m": float(np.percentile(err, 95)),
        "share_err_gt300m": float(np.average(err > 300, weights=w_all)),
    })
    print(f"  {label:14s} {len(sizes):5d} clusters, max {sizes.max():3d} blocks, mean err {np.average(err, weights=w_all):5.0f} m")
res = pd.DataFrame(rows)

  dbscan_150m      253 clusters, max 413 blocks, mean err   339 m


  dbscan_200m      207 clusters, max 413 blocks, mean err   355 m
  dbscan_250m      192 clusters, max 413 blocks, mean err   367 m


  dbscan_300m      182 clusters, max 413 blocks, mean err   372 m


  complete_250m   1762 clusters, max  23 blocks, mean err    67 m


  complete_300m   1403 clusters, max  31 blocks, mean err    79 m


  complete_400m    984 clusters, max  40 blocks, mean err   101 m
  complete_500m    737 clusters, max  40 blocks, mean err   123 m


  average_200m    1283 clusters, max  34 blocks, mean err    83 m
  average_300m     714 clusters, max  55 blocks, mean err   125 m


  average_400m     474 clusters, max  64 blocks, mean err   167 m


## 4. Results

Read `mean_err_m` as the walking distance misstated per flat if one walk time per cluster were used
(80 m is about one minute). DBSCAN's 95th-percentile diameter is in kilometres; complete linkage at
300 m keeps it under the cap by construction.

In [5]:
res.to_csv(OUT_CSV, index=False)
pd.set_option("display.width", 250)
print(res.round(0).to_string(index=False))
print(f"\nWrote {OUT_CSV.relative_to(ROOT)} ({len(res)} configurations)")

       config                                family  param_m  n_clusters  singletons  median_blocks  max_blocks  median_units  max_units  median_diam_m  p95_diam_m  max_diam_m  mean_err_m  p95_err_m  share_err_gt300m
  dbscan_150m         DBSCAN (single-link chaining)      150         253          10           21.0         413        2806.0      37486          524.0      1611.0      2395.0       339.0      770.0               0.0
  dbscan_200m         DBSCAN (single-link chaining)      200         207           5           33.0         413        4322.0      37486          693.0      1714.0      2395.0       355.0      771.0               1.0
  dbscan_250m         DBSCAN (single-link chaining)      250         192           4           37.0         413        4510.0      37486          775.0      1736.0      2681.0       367.0      790.0               1.0
  dbscan_300m         DBSCAN (single-link chaining)      300         182           3           42.0         413        4794.0      3